In [1]:
import os 

In [2]:
%pwd

'd:\\a27_YEARS_OLD\\nlp_text_summarization\\research'

In [3]:
os.chdir("../")

In [4]:
%pwd

'd:\\a27_YEARS_OLD\\nlp_text_summarization'

In [5]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class DataTransformationConfig:
    root_dir: Path
    data_path: Path
    tokenizer_name: Path

In [6]:
from textSummarizer.constants import *
from textSummarizer.utils.common import read_yaml, create_directories

In [7]:
class ConfigurationManager:
    def __init__(
            self,
            config_filepath = CONFIG_FILE_PATH,
            params_filepath = PARAMS_FILE_PATH):
        
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)

        create_directories([self.config.artifacts_root])


    def get_data_transformation_config(self) -> DataTransformationConfig:
        config = self.config.data_transformation

        create_directories([config.root_dir])

        data_transformation_config = DataTransformationConfig(
            root_dir=config.root_dir, # artifacts/data_transformation
            data_path=config.data_path, # artifacts/data_ingestion/samsum_dataset
            tokenizer_name=config.tokenizer_name # google/pegasus-cnn_dailymail
        )

        return data_transformation_config

In [9]:
import os
from textSummarizer.logging import logger
from transformers import AutoTokenizer
from datasets import load_dataset, load_from_disk

In [10]:
class DataTransformation:
    def __init__(self, config: DataTransformationConfig):
        self.config = config
        self.tokenizer = AutoTokenizer.from_pretrained(config.tokenizer_name) # google/pegasus-cnn_dailymail


    def convert_examples_to_features(self,example_batch):# example_batch - samsum data csv file (id,dialog, summary)
        input_encodings = self.tokenizer(example_batch['dialogue'] , max_length = 1024, truncation = True )

        with self.tokenizer.as_target_tokenizer():
            target_encodings = self.tokenizer(example_batch['summary'], max_length = 128, truncation = True )

        return { # feature dictionary
            'input_ids' : input_encodings['input_ids'],
            'attention_mask': input_encodings['attention_mask'], # [START, END]
            'labels': target_encodings['input_ids']
    }

    def convert(self):
        dataset_samsum = load_from_disk(self.config.data_path) # artifacts/data_ingestion/samsum_dataset
        dataset_samsum_pt = dataset_samsum.map(self.convert_examples_to_features, batched=True)
        dataset_samsum_pt.save_to_disk(os.path.join(self.config.root_dir, "samsum_dataset")) # artifacts/data_transformation


In [ ]:
# The map method from the datasets library is utilized to apply the convert_examples_to_features function 
# to the entire samsum dataset. The batched=True argument ensures that the function is applied batch-wise, 
# which can be more efficient than processing each example individually.

In [1]:
# This method is converting the text data in example_batch into a format that can be used as input to a transformer-based model.
# Here's what's happening:
# Input Encoding
# Tokenization: The dialogue column in example_batch is passed through the tokenizer to split the text into subwords (smaller units of text, such as word pieces).
# Encoding: The tokenized text is then encoded into integers using the tokenizer's vocabulary.
# Padding/Truncation: The encoded text is padded or truncated to a maximum length of 1024 tokens.
# The resulting encoding is stored in the input_encodings dictionary.
# Target Encoding
# Target Tokenizer: The tokenizer is used as a target tokenizer, which is a tokenizer that is specifically designed for the target text (in this case, the summary column).
# Tokenization: The summary column in example_batch is passed through the target tokenizer to split the text into subwords.
# Encoding: The tokenized text is then encoded into integers using the target tokenizer's vocabulary.
# Padding/Truncation: The encoded text is padded or truncated to a maximum length of 128 tokens.
# The resulting encoding is stored in the target_encodings dictionary.
# Feature Dictionary
# The method returns a dictionary with the following features:
# input_ids: The encoded input text (dialogue)
# attention_mask: A mask indicating which tokens in the input text are actual tokens and which are padding tokens
# labels: The encoded target text (summary)
# These features can be used as input to a transformer-based model for training.

In [ ]:
# The encoding method used here is likely WordPiece tokenization, which is a subword tokenization algorithm.
# WordPiece tokenization is a technique used to represent words as a sequence of subwords, or word pieces. This allows the model to handle out-of-vocabulary (OOV) words and to capture subtle differences in word meanings.
# Here's how it works:
# Vocabulary creation: A vocabulary of word pieces is created from a large corpus of text.
# Tokenization: When a word is encountered, it is split into its constituent word pieces.
# Encoding: Each word piece is assigned a unique integer ID, which is used to represent the word piece in the model.
# In the context of the convert_examples_to_features method, the tokenizer object is likely an instance of the BertTokenizer class, which uses WordPiece tokenization.
# The input_encodings and target_encodings dictionaries contain the encoded input and target text, respectively, where each token is represented by its corresponding integer ID.
# For example, the input text "This is an example sentence" might be encoded as:
# [101, 2023, 2003, 1037, 3286, 2023, 2003, 1037, 3286, 1029]
# Where each integer corresponds to a word piece in the vocabulary.

In [ ]:
# BERT (Bidirectional Encoder Representations from Transformers) is a pre-trained language model 
# developed by Google that has revolutionized the field of natural language processing (NLP).

In [ ]:
# The load_from_disk function is likely from the Hugging Face datasets library, which is used to load a dataset from a file on disk.
# In this case, the dataset_samsum variable is loaded from a file located at self.config.data_path, which is set to "artifacts/data_ingestion/samsum_dataset".
# The SAMSum dataset is a collection of conversations and their corresponding summaries. It is a popular dataset for training and evaluating conversational summarization models.
# The data is likely stored in a format such as:
# JSON: Each conversation and summary is stored as a JSON object, with keys for the conversation ID, conversation text, and summary text.
# CSV: Each conversation and summary is stored as a row in a CSV file, with columns for the conversation ID, conversation text, and summary text.
# Arrow: Each conversation and summary is stored in an Apache Arrow file, which is a columnar storage format.
# The load_from_disk function can handle various data formats, including JSON, CSV, and Arrow.
# Once loaded, the dataset_samsum variable is a Dataset object, which provides various methods for manipulating and processing the data, such as filtering, mapping, and batching.


In [ ]:
# samsum/
# data/
# train.json
# validation.json
# test.json
# ...


# or

# artifacts/
# data_ingestion/
# samsum_dataset/
# train.arrow
# validation.arrow
# test.arrow
# ...


In [12]:
try:
    config = ConfigurationManager()
    data_transformation_config = config.get_data_transformation_config()
    data_transformation = DataTransformation(config=data_transformation_config)
    data_transformation.convert()
except Exception as e:
    raise e

[2024-09-19 22:09:45,130: INFO: common: yaml file: config\config.yaml loaded successfully]
[2024-09-19 22:09:45,132: INFO: common: yaml file: params.yaml loaded successfully]
[2024-09-19 22:09:45,133: INFO: common: created directory at: artifacts]
[2024-09-19 22:09:45,135: INFO: common: created directory at: artifacts/data_transformation]


d:\a27_YEARS_OLD\nlp_text_summarization\venv\Lib\site-packages\transformers\tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(
Map:   0%|          | 0/14732 [00:00<?, ? examples/s]d:\a27_YEARS_OLD\nlp_text_summarization\venv\Lib\site-packages\transformers\tokenization_utils_base.py:4126: UserWarning: `as_target_tokenizer` is deprecated and will be removed in v5 of Transformers. You can tokenize your labels by using the argument `text_target` of the regular `__call__` method (either in the same call as your input texts if you use the same keyword arguments, or in a separate call.
  warnings.warn(
Saving the dataset (1/1 shards): 100%|██████████| 818/818 [00:00<00:00, 87126.15 examples/s] 


coco dataset for image

In [ ]:
# The COCO (Common Objects in Context) dataset is a large-scale dataset for object detection, segmentation, and captioning tasks.
# Key Features:
# Images: The dataset consists of over 330,000 images, each with an average of 7 objects.
# Object Instances: The dataset contains over 2.5 million object instances, each labeled with a class label and bounding box coordinates.
# Object Classes: The dataset includes 91 object classes, such as person, car, dog, and chair.
# Segmentation Masks: The dataset provides pixel-level segmentation masks for each object instance.
# Captions: The dataset includes over 500,000 captions, each describing the content of an image.
# Tasks:
# Object Detection: Detecting and classifying objects within images.
# Object Segmentation: Segmenting objects from the background and other objects.
# Image Captioning: Generating captions that describe the content of an image.
# Evaluation Metrics:
# AP (Average Precision): Measures the average precision of object detection.
# AR (Average Recall): Measures the average recall of object detection.
# BLEU Score: Measures the quality of generated captions.
# Applications:
# Computer Vision: COCO is widely used in computer vision research and applications.
# Robotics: COCO is used in robotics to enable robots to understand and interact with their environment.
# Autonomous Vehicles: COCO is used in autonomous vehicles to detect and recognize objects on the road.

In [ ]:
# The COCO dataset is stored in several formats, including:
# 1. JSON
# The annotations are stored in JSON files, which contain the following information:
# Image metadata (e.g., image ID, width, height)
# Object instances (e.g., object ID, class label, bounding box coordinates)
# Segmentation masks (e.g., pixel-level masks for each object instance)
# 2. PNG/JPEG Images
# The images themselves are stored as PNG or JPEG files.
# 3. Binary Masks (optional)
# The segmentation masks can also be stored as binary masks, which are used for training and evaluating segmentation models.
# 4. Pkl Files (optional)
# Some pre-processed versions of the COCO dataset are stored in Python pickle (pkl) files, which contain pre-computed features and annotations.
# Directory Structure:
# The COCO dataset is typically organized in the following directory structure:
# coco/
# annotations/
# instances_train2014.json
# instances_val2014.json
# ...
# images/
# train2014/
# val2014/
# ...
# This structure contains separate directories for the annotations and images, with further subdirectories for the different splits (e.g., train, val).